In [ ]:
# --- scikit-learn 版本的多层感知机（MLP）实现 ---
from sklearn.neural_network import MLPClassifier  # 多层感知机分类器
from sklearn.datasets import fetch_openml
import numpy as np

# 加载 MNIST 手写数字数据集（70000张28x28灰度图像）
mnist = fetch_openml('mnist_784')
X, y = mnist['data'], mnist['target']
# 按经典划分：前60000张训练，后10000张测试
X_train = np.array(X[:60000], dtype=float)
y_train = np.array(y[:60000], dtype=float)
X_test = np.array(X[60000:], dtype=float)
y_test = np.array(y[60000:], dtype=float)

# 构建 MLP 分类器
# alpha=1e-5：L2 正则化系数，防止过拟合
# hidden_layer_sizes=(15,15)：两层隐藏层，每层15个神经元
# MLP 结构：784(输入) -> 15 -> 15 -> 10(输出，对应10个数字类别)
# 前向传播：h = activation(W*x + b)，逐层计算直到输出层
# 反向传播：通过链式法则计算梯度，更新权重以最小化损失函数
clf = MLPClassifier(alpha=1e-5,
                    hidden_layer_sizes=(15,15), random_state=1)

clf.fit(X_train, y_train)  # 训练模型，使用反向传播算法优化权重

score = clf.score(X_test, y_test)  # 在测试集上计算准确率

In [3]:
score

0.7791

参数意义：

hidden_layer_sizes :隐藏层大小，(50,50)表示有两层隐藏层，第一层隐藏层有50个神经元，第二层也有50个神经元。

activation :激活函数,{‘identity’, ‘logistic’, ‘tanh’, ‘relu’}, 默认为relu

solver： 权重优化器，{‘lbfgs’, ‘sgd’, ‘adam’}, 默认adam

learning_rate_int:double,可选，默认0.001，初始学习率，控制更新权重的补偿，只有当solver=’sgd’ 或’adam’时使用。


In [ ]:
# --- PyTorch 数据加载 + Logistic 回归对比实验 ---
import sys
from pathlib import Path
curr_path = str(Path().absolute()) 
parent_path = str(Path().absolute().parent) 
sys.path.append(parent_path) 

# 添加目录到系统路径方便导入模块，该项目的根目录为".../machine-learning-toy-code"
from pathlib import Path
from torch.utils.data import DataLoader  # PyTorch 数据加载器
from torchvision import datasets
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import numpy as np

curr_path = str(Path().absolute())
parent_path = str(Path().absolute().parent)
p_parent_path = str(Path().absolute().parent.parent)
sys.path.append(p_parent_path) 
print(f"主目录为：{p_parent_path}")

# 通过 torchvision 加载 MNIST 数据集
# transforms.ToTensor() 将图像像素值从 [0,255] 转换为 [0,1] 的 Tensor
train_dataset = datasets.MNIST(root = p_parent_path+'/datasets/', train = True,transform = transforms.ToTensor(), download = False)
test_dataset = datasets.MNIST(root = p_parent_path+'/datasets/', train = False, 
                               transform = transforms.ToTensor(), download = False)

# 一次性加载全部数据（batch_size=整个数据集大小）
batch_size = len(train_dataset)
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=True)
X_train,y_train = next(iter(train_loader))  # 获取训练数据
X_test,y_test = next(iter(test_loader))     # 获取测试数据
# 可视化前100张图片，直观查看数据集
images, labels= X_train[:100], y_train[:100] 
img = torchvision.utils.make_grid(images, nrow=10)  # 生成10列网格图
# Tensor 格式为 (channels, H, W)，需转为 (H, W, channels) 才能用 plt 显示
img = img.numpy().transpose(1,2,0)
print(images.shape)
print(labels.reshape(10,10))
print(img.shape)
plt.imshow(img)
plt.show()

# 将 PyTorch Tensor 转为 NumPy 数组，供 scikit-learn 使用
X_train,y_train = X_train.cpu().numpy(),y_train.cpu().numpy()
X_test,y_test = X_test.cpu().numpy(),y_test.cpu().numpy()

# 将 (N, 1, 28, 28) 的图像展平为 (N, 784) 的一维特征向量
X_train = X_train.reshape(X_train.shape[0],784)
X_test = X_test.reshape(X_test.shape[0],784)

# --- 多分类 Logistic 回归（作为 MLP 的对比基线） ---
# solver="lbfgs"：拟牛顿法优化器，适合中小规模数据集
model = LogisticRegression(solver='lbfgs', max_iter=400)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))  # 输出精确率、召回率、F1分数

# --- 二分类 Logistic 回归（判断是否为数字"1"） ---
# 为特征矩阵添加偏置列（全1），使线性组合可包含截距项
ones_col=[[1] for i in range(len(X_train))]
X_train = np.append(X_train,ones_col,axis=1)
x_train = np.mat(X_train)
X_test = np.append(X_test,ones_col,axis=1)
x_test = np.mat(X_test)
# 将多标签转化为二分类标签：数字"1"为正类，其余为负类
y_train=np.array([1 if y_train[i]==1 else 0 for i in range(len(y_train))])
y_test=np.array([1 if y_test[i]==1 else 0 for i in range(len(y_test))])

# 二分类 Logistic 回归
model = LogisticRegression(solver='lbfgs', max_iter=100)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))  # 打印评估报告